<a id="top"></a>
# How to Browse Roman Observations
***
## Learning Goals

This is a beginner tutorial on browsing observations and data products from
the Roman Space Telescope available on MAST. We'll explore Roman observation
metadata and viewing results interactively. By the end of this tutorial, you
will:

- Understand the structure of Roman observation metadata.
- Be able to view and explore observation query results with `mast-table`.
- Know how to visualize observation footprints with `mast-aladin`.

## Introduction

This tutorial shows you how to explore Roman observations from MAST
interactively in a Jupyter notebook.

You'll use two Python packages:
- [`mast-table`](https://github.com/spacetelescope/mast-table) to view
  observation data in an interactive table.
- [`mast-aladin`](https://github.com/spacetelescope/mast-aladin) to
  visualize observation footprints on a sky map.

## About the Data

**Note:** Roman Space Telescope observations are not yet publicly available. This tutorial uses
representative cached data from simulated Roman observations to demonstrate the interactive visualization
features of `mast-table` and `mast-aladin`.

## Imports

We will import the following in this tutorial:

- `astropy.table.Table`: work with tabular data in ECSV format.
- `mast_table.MastTable`: interactive widget for browsing observation
  data.
- `mast_aladin.MastAladin`: interactive sky map widget for visualizing
  observation footprints.

In [ ]:
from astropy.table import Table

from mast_aladin import MastAladin
from mast_table import MastTable

***

## Loading Roman Observation Data

In this section, we load cached Roman observation data and explore the available metadata columns.

### Searchable Column List

First, let's load the list of searchable columns available in the Roman metadata:

In [ ]:
# When Roman data are publicly available in the archive, you can retrieve the
# column list like this:
# from astroquery.mast import MastMissions
# mast = MastMissions(mission="roman")
# columns = mast.get_column_list()

# Until then, we load representative Roman query results from cache:
columns = Table.read("data/roman_columns.ecsv", format="ascii.ecsv")
print(f"Found {len(columns)} searchable columns. First 10:\n")
for row in columns[:10]:
    print(f"  {row['name']:30s} ({row['data_type']})")

### Loading Observation Data

Now load the cached observation data. This dataset contains observations from
RA=270°, Dec=66° with the F146 filter (a Roman WFI wide-band filter). The data
includes all available metadata columns, including the `s_region` polygon used
for footprint visualization.

In [ ]:
# When Roman data are publicly available in the archive, you can query for
# observations like this:
# import astropy.units as u
# from astropy.coordinates import SkyCoord
# target = SkyCoord(ra=270, dec=66, unit="deg")
# roman_obs = mast.query_region(
#     target,
#     radius=0.3 * u.deg,
#     optical_element="F146",
#     select_cols="*",
# )

# Until then, we load representative Roman query results from cache:
roman_obs = Table.read("data/roman_obs_f146.ecsv", format="ascii.ecsv")

print(f"Found {len(roman_obs)} Roman observations with F146 filter")
roman_obs[:5]

**Observations across product levels.** This dataset includes all processing
levels (1=raw, 2=calibrated, 3=resampled, 4=high-level) for the same sky
region:

In [ ]:
# When Roman data are publicly available in the archive, query across all
# product levels:
# all_obs = mast.query_criteria(
#     coordinates=target,
#     radius=0.3 * u.deg,
#     productLevel=["1", "2", "3", "4"],
#     select_cols="*",
# )

# Until then, we load representative Roman query results from cache:
all_obs = Table.read("data/all_obs.ecsv", format="ascii.ecsv")

print(f"Found {len(all_obs)} observations across all processing levels")
all_obs[:3]

The results are returned as an [`astropy.table.Table`](https://docs.astropy.org/en/stable/table/).
Useful columns include:

- `fileSetName`: unique identifier for the observation file set.
- `productLevel`: processing level (1=raw, 2=calibrated, 3=resampled,
  4=high-level).
- `optical_element`: filter used (F062, F087, F106, F129, F146, F158,
  F184 for WFI).
- `exposure_time`: exposure time in seconds.
- `detector`: detector identifier (WFI01 to WFI18 for the WFI mosaic).
- `program`: program ID number.
- `s_region`: sky-region footprint as an STC-S `POLYGON` string.

### Another Sky Position

Here's another dataset from a different sky position near the galactic center (RA=266.4°, Dec=-29.0°):

In [ ]:
# When Roman data are publicly available in the archive, query near the
# galactic center:
# galactic_center = SkyCoord(ra=266.4, dec=-29.0, unit="deg")
# position_obs = mast.query_criteria(
#     coordinates=galactic_center,
#     radius=0.5 * u.deg,
#     productLevel=["1", "2", "3", "4"],
#     select_cols="*",
# )

# Until then, we load representative Roman query results from cache:
position_obs = Table.read("data/position_obs.ecsv", format="ascii.ecsv")

print(f"Found {len(position_obs)} observations near the galactic center")
position_obs[:5]

***

## Viewing Results with mast-table

`mast-table` is an interactive Jupyter widget for browsing observation
metadata tables. It supports pagination, column sorting, search-based
filtering, and column visibility toggles: all without leaving the
notebook.

### Creating an Interactive Table

Pass the Astropy table to `MastTable` to create an interactive widget:

In [ ]:
# Create an interactive table widget from the observation data.
table_widget = MastTable(roman_obs)
table_widget

The table widget displays above with several interactive features:
- **Pagination controls** at the bottom right to navigate through pages of results.
- **Column sorting**: click on a column header to sort by that column.
- **Search box**: filter rows by typing in the search field.
- **Column visibility**: toggle which columns are displayed.

### Exploring Table Features

Try the following to explore the table's capabilities:

1. **Change pages.** Use the arrow buttons at the bottom right to view different pages of results.
2. **Sort data.** Click on the `optical_element` column header to sort observations by filter.
3. **Search.** Try typing a program ID (for example, `185`) in the search box to filter the table.
4. **Adjust page size.** Use the "Rows per page" dropdown to display more or fewer results at once.

For a deeper dive on customizing `mast-table`, see the companion notebook Inspect Long Tables.

***

## Visualizing Footprints with mast-aladin

For a spatial view of the observations, `mast-aladin` renders an interactive
sky map (powered by Aladin Lite) directly in the notebook. This is
particularly useful for understanding the sky coverage of Roman observations
and how they overlap.

### Loading Footprints

Create an Aladin widget centered on the target region (RA=270°, Dec=66°).
The sky map can then be overlaid with the polygon footprint of each Roman
observation.

In [ ]:
# Create an Aladin widget centered on the target region (RA=270, Dec=66).
aladin = MastAladin(
    target="270 66",
    fov=1.0,  # Field of view in degrees.
    height=500,
)
aladin

### Interactive Sky View

The Aladin widget provides several interactive features:

- **Pan.** Click and drag to move around the sky.
- **Zoom.** Use the scroll wheel or the zoom controls.
- **Layers.** Toggle different survey layers (e.g., DSS, 2MASS) using the
  layer controls.
- **Footprints.** Observation footprints can be displayed as polygons.

Roman footprints are stored in the `s_region` column as STC-S `POLYGON`
strings, which `mast-aladin` can overlay directly:

In [ ]:
# Overlay Roman observation footprints on the Aladin sky map.
if len(roman_obs) > 0 and "s_region" in roman_obs.colnames:
    aladin.add_graphic_overlay_from_stcs(
        roman_obs["s_region"],
        color="cyan",
        opacity=0.5,
    )
    print(f"Added {len(roman_obs)} observation footprints to the map")
else:
    print("No footprint data available")

The cyan polygons show the actual sky coverage of each Roman observation in
the dataset. Scroll back to the Aladin widget above to inspect them: you can
pan, zoom, and toggle background surveys to see how Roman tiles overlap real
reference imagery. For more options, see the
[`mast-aladin` documentation](https://github.com/spacetelescope/mast-aladin).

***

## Additional Resources

- [Roman Space Telescope mission site](https://roman.gsfc.nasa.gov/):
  official mission information.
- [MAST Roman page](https://archive.stsci.edu/missions-and-data/roman):
  archive landing page for Roman.
- [`astroquery.mast.MastMissions` documentation](https://astroquery.readthedocs.io/en/latest/mast/mast_missions.html):
  API reference for mission-specific MAST queries.
- [`mast-table` repository](https://github.com/spacetelescope/mast-table):
  widget documentation and examples.
- [`mast-aladin` repository](https://github.com/spacetelescope/mast-aladin):
  interactive sky-visualization documentation.
- [MAST Search API](https://mast.stsci.edu/search/docs/):
  underlying REST API used by `MastMissions`.
- Companion notebook: Inspect Long Tables: deeper dive into customizing
  `mast-table` views.

## Citations

If you use `astroquery`, `astropy`, `mast-table`, or `mast-aladin` for
published research, please cite the authors. Follow these links for more
information:

- [Citing `astroquery`](https://github.com/astropy/astroquery/blob/main/astroquery/CITATION)
- [Citing `astropy`](https://www.astropy.org/acknowledging.html)
- [Citing `mast-table`](https://github.com/spacetelescope/mast-table)
- [Citing `mast-aladin`](https://github.com/spacetelescope/mast-aladin)

## About This Notebook

**Author:** Hatice Karatay <br>
**Keywords:** Roman, MAST, astroquery, mast-table, mast-aladin,
observations <br>
**First published:** July 2026 <br>
**Last updated:** July 2026 <br>

***
[Top of Page](#top)
<img style="float: right;"
     src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png"
     alt="Space Telescope Logo" width="200px"/>